# Anomaly Detection Modeling Report (Binary + Multiclass)

**Abstract** — This notebook trains and evaluates supervised models for (1) binary anomaly detection (Normal vs. Anomaly) and (2) multiclass classification (per anomaly type) from CSV telemetry data. It applies leakage guards, uses numeric features only, and compares linear and tree-based scikit-learn pipelines with balanced class weighting. A simple Keras DNN is executed if TensorFlow is available.


## Overview
This report-style notebook implements an end-to-end training/evaluation workflow on a CSV dataset:

- **Binary task**: predict `anomaly.label` (Normal vs Anomaly)
- **Multiclass task**: predict `anomaly.layer` (anomaly category)

The workflow is designed for reproducibility and repository inclusion: deterministic seeding, explicit preprocessing pipelines, and consistent metrics reporting.


## Setup
The environment relies on **NumPy**, **pandas**, and **scikit-learn**. If **TensorFlow/Keras** is installed, an optional lightweight DNN baseline is also run.


In [ ]:
# -*- coding: utf-8 -*-
from __future__ import annotations

import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt


## Methodology
### Data ingestion and label handling
Data can be loaded from a single CSV file or a directory of CSVs matched via a glob pattern. Rows with missing target labels are dropped. The binary label is normalized to `0/1` to support heterogeneous encodings (numeric or string-based).

### Feature selection and leakage guard
Only numeric columns are used as features. In addition, `anomaly.*` metadata columns (except the targets) and common identifier/timestamp-like fields are excluded to reduce the risk of label leakage.

### Modeling
Two preprocessing strategies are used:
- **Linear/DNN**: median imputation + standard scaling
- **Tree-based**: median imputation only

Class imbalance is handled via **balanced class weights**. Since AdaBoost does not consistently respect `class_weight`, equivalent **sample weights** are passed explicitly during fitting.

### Evaluation
Binary metrics include accuracy, classification report, confusion matrix, and ROC-AUC (when probabilities are available). Multiclass metrics include accuracy, macro-averaged classification report, and confusion matrix.


In [ ]:
# =====================
# Paths / configuration
# =====================
INPUT_PATH = Path(".")      # directory with CSVs OR a single CSV file
GLOB = "*_v11.csv"          # ignored if INPUT_PATH is a file

TARGET_COL = "anomaly.layer"    # multiclass label
BINARY_COL = "anomaly.label"    # binary label (0/1 or strings like 'Normal'/'Anomaly')

TEST_SIZE = 0.2
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)


In [ ]:
def load_csvs(input_path: Path, glob_pattern: str) -> pd.DataFrame:
    """Load one CSV or concatenate multiple CSVs from a directory.

    Parameters
    ----------
    input_path:
        A single CSV file path or a directory containing CSVs.
    glob_pattern:
        Glob pattern used when `input_path` is a directory.

    Returns
    -------
    pd.DataFrame
        The loaded dataset.
    """
    if input_path.is_file():
        return pd.read_csv(input_path, low_memory=False)

    frames = []
    for p in sorted(input_path.glob(glob_pattern)):
        try:
            frames.append(pd.read_csv(p, low_memory=False))
        except Exception as exc:
            print(f"[WARN] Skipping {p}: {exc}")

    if not frames:
        raise SystemExit("No CSV files loaded. Check INPUT_PATH/GLOB.")

    return pd.concat(frames, ignore_index=True)


raw = load_csvs(INPUT_PATH, GLOB)

# Drop rows missing labels
raw = raw.dropna(subset=[TARGET_COL, BINARY_COL])

# Clean multiclass target
raw[TARGET_COL] = raw[TARGET_COL].astype(str).str.strip()

# Prepare binary label from BINARY_COL
if pd.api.types.is_numeric_dtype(raw[BINARY_COL]):
    raw[BINARY_COL] = (raw[BINARY_COL].astype(float) > 0).astype(int)
else:
    s = raw[BINARY_COL].astype(str).str.strip().str.lower()
    benign = {"0", "normal", "benign", "ok", "none", "no_anomaly"}
    raw[BINARY_COL] = (~s.isin(benign)).astype(int)

raw.shape


## Implementation
This section applies the leakage guard, builds train/test splits for the binary and multiclass tasks, defines preprocessors and class weights, and then fits comparable model families (Logistic Regression, Random Forest, AdaBoost).

In [ ]:
# =====================
# Features (leak guard + numeric only)
# =====================

exclude_for_multi = {TARGET_COL} if TARGET_COL in raw.columns else set()
exclude_for_bin = {BINARY_COL} if BINARY_COL in raw.columns else set()
exclude_cols = exclude_for_multi.union(exclude_for_bin)

# Exclude any other anomaly.* metadata that could leak labels
leak_guard = [
    c for c in raw.columns
    if c.startswith("anomaly.") and c not in {TARGET_COL, BINARY_COL}
]
exclude_cols = exclude_cols.union(leak_guard)

# Exclude common identifier/timestamp-like columns
maybe_id_cols = {
    "frame.time",
    "ip.src_host",
    "ip.dst_host",
    "tcp.payload",
    "http.file_data",
    "mqtt.msg",
    "session_id",
    "timestamp",
    "time",
    "Time",
    "tcp.dstport",
    "tcp.srcport",
    "Datetime",
    "datetime",
    "Unnamed: 0",
    "MAC_extracted",
}

print(
    "Excluding columns (potential leakage/IDs):",
    sorted(list(exclude_cols.union(maybe_id_cols))),
)

feature_df = raw.drop(columns=list(exclude_cols.union(maybe_id_cols)), errors="ignore")
num_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()

if not num_cols:
    raise SystemExit("No numeric feature columns found after leak guard and filtering.")

X = feature_df[num_cols].copy()

# Targets
y_multi = raw[TARGET_COL].copy()
y_bin = raw[BINARY_COL].copy()

# Sanity: ensure no label columns in features
assert TARGET_COL not in X.columns and BINARY_COL not in X.columns, "Label leakage into X!"

X.shape, y_bin.value_counts(dropna=False), y_multi.nunique()


In [ ]:
# =====================
# Split (separate splits per task)
# =====================

X_train_bin, X_test_bin, y_bin_train, y_bin_test = train_test_split(
    X,
    y_bin,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_bin,
)

X_train_mc, X_test_mc, y_multi_train, y_multi_test = train_test_split(
    X,
    y_multi,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_multi,
)

(X_train_bin.shape, X_test_bin.shape, X_train_mc.shape, X_test_mc.shape)


In [ ]:
# =====================
# Preprocessors
# =====================

# Linear/DNN: scaling helps
lin_num_processor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ]
)
lin_preprocess = ColumnTransformer([("num", lin_num_processor, num_cols)])

# Trees (RF/AdaBoost): no scaling needed
tree_num_processor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)
tree_preprocess = ColumnTransformer([("num", tree_num_processor, num_cols)])


In [ ]:
# =====================
# Class weights
# =====================

classes_multi = np.unique(y_multi_train)
class_weights_multi = compute_class_weight(
    class_weight="balanced",
    classes=classes_multi,
    y=y_multi_train,
)
class_weight_dict_multi = {c: float(w) for c, w in zip(classes_multi, class_weights_multi)}

classes_bin = np.unique(y_bin_train)
class_weights_bin = compute_class_weight(
    class_weight="balanced",
    classes=classes_bin,
    y=y_bin_train,
)
class_weight_dict_bin = {int(c): float(w) for c, w in zip(classes_bin, class_weights_bin)}

print("Class weights (binary) [label -> weight]:", class_weight_dict_bin)


### Evaluation helpers
The helpers below standardize metric reporting and add lightweight visualization for confusion matrices. The plotting is intentionally minimal to keep outputs repository-friendly.

In [ ]:
def _plot_confusion(cm: np.ndarray, title: str, class_names: list[str]) -> None:
    """Plot a confusion matrix using matplotlib."""
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha="center", va="center")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


def eval_binary(model_name: str, y_true, y_pred, y_proba=None) -> dict:
    """Evaluate and print standard binary classification metrics."""
    print(f"=== [BINARY] {model_name} ===")
    acc = accuracy_score(y_true, y_pred)
    print("Accuracy:", acc)
    print("Classification report (pos=1):")
    print(classification_report(y_true, y_pred, digits=4))

    auc = None
    if y_proba is not None:
        try:
            auc = roc_auc_score(y_true, y_proba)
            print("ROC-AUC:", auc)
        except Exception as exc:
            print("ROC-AUC not available:", exc)

    cm = confusion_matrix(y_true, y_pred)
    print("Confusion matrix:")
    print(cm)
    _plot_confusion(cm, f"[BINARY] {model_name} — Confusion Matrix", ["0", "1"])

    return {"model": model_name, "accuracy": acc, "roc_auc": auc}


def eval_multiclass(model_name: str, y_true, y_pred) -> dict:
    """Evaluate and print standard multiclass classification metrics."""
    print(f"=== [MULTICLASS] {model_name} ===")
    acc = accuracy_score(y_true, y_pred)
    print("Accuracy:", acc)
    print("Macro avg report:")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    labels = sorted(pd.unique(pd.Series(list(y_true) + list(y_pred)).astype(str)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print("Confusion matrix:")
    print(cm)
    _plot_confusion(cm, f"[MULTICLASS] {model_name} — Confusion Matrix", labels)

    return {"model": model_name, "accuracy": acc}


def compute_sample_weights(y, class_weight_dict) -> np.ndarray:
    """Compute per-sample weights from a class_weight mapping.

    Notes
    -----
    AdaBoost does not reliably consume `class_weight`, so sample weights are passed
    directly to the estimator during `fit()`.
    """
    weights = []
    for c in y:
        if c in class_weight_dict:
            weights.append(class_weight_dict[c])
        else:
            try:
                weights.append(class_weight_dict[int(c)])
            except Exception:
                weights.append(1.0)
    return np.asarray(weights, dtype=float)


### Binary models
Three baselines are trained:
- **Logistic Regression**: linear model with scaling; provides calibrated probabilities for ROC-AUC.
- **Random Forest**: non-linear ensemble; robust to feature scaling.
- **AdaBoost (SAMME)**: configured with deeper base trees to remain competitive and uses explicit sample weights.


In [ ]:
# =====================
# Models — Binary
# =====================

logreg_bin = Pipeline(
    steps=[
        ("prep", lin_preprocess),
        ("clf", LogisticRegression(max_iter=2000, class_weight=class_weight_dict_bin)),
    ]
)

rf_bin = Pipeline(
    steps=[
        ("prep", tree_preprocess),
        (
            "clf",
            RandomForestClassifier(
                n_estimators=300,
                random_state=RANDOM_STATE,
                class_weight=class_weight_dict_bin,
                n_jobs=-1,
            ),
        ),
    ]
)

# Fairer AdaBoost (binary): deeper base learner + SAMME
ada_bin = Pipeline(
    steps=[
        ("prep", tree_preprocess),
        (
            "clf",
            AdaBoostClassifier(
                estimator=DecisionTreeClassifier(max_depth=2, min_samples_leaf=5),
                n_estimators=400,
                learning_rate=0.5,
                algorithm="SAMME",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

sw_bin = compute_sample_weights(y_bin_train.values, class_weight_dict_bin)

# Fit
logreg_bin.fit(X_train_bin, y_bin_train)
rf_bin.fit(X_train_bin, y_bin_train)
ada_bin.fit(X_train_bin, y_bin_train, clf__sample_weight=sw_bin)

# Predict / evaluate
try:
    bin_proba = logreg_bin.predict_proba(X_test_bin)[:, 1]
except Exception:
    bin_proba = None

bin_results = []
bin_results.append(eval_binary("LogReg", y_bin_test, logreg_bin.predict(X_test_bin), bin_proba))
bin_results.append(eval_binary("RandomForest", y_bin_test, rf_bin.predict(X_test_bin)))
bin_results.append(eval_binary("AdaBoost", y_bin_test, ada_bin.predict(X_test_bin)))

pd.DataFrame(bin_results)


### Multiclass models
The same model families are trained for the multiclass task. AdaBoost is configured with deeper trees, SAMME, more estimators, and a lower learning rate to better handle multi-class settings.


In [ ]:
# =====================
# Models — Multiclass
# =====================

logreg_multi = Pipeline(
    steps=[
        ("prep", lin_preprocess),
        (
            "clf",
            LogisticRegression(
                max_iter=2000,
                class_weight=class_weight_dict_multi,
                multi_class="auto",
            ),
        ),
    ]
)

rf_multi = Pipeline(
    steps=[
        ("prep", tree_preprocess),
        (
            "clf",
            RandomForestClassifier(
                n_estimators=300,
                random_state=RANDOM_STATE,
                class_weight=class_weight_dict_multi,
                n_jobs=-1,
            ),
        ),
    ]
)

# Fair multiclass AdaBoost: deeper trees + SAMME + more estimators + lower LR
ada_multi = Pipeline(
    steps=[
        ("prep", tree_preprocess),
        (
            "clf",
            AdaBoostClassifier(
                estimator=DecisionTreeClassifier(max_depth=3, min_samples_leaf=10),
                n_estimators=600,
                learning_rate=0.3,
                algorithm="SAMME",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

sw_multi = np.array([class_weight_dict_multi.get(c, 1.0) for c in y_multi_train.values], dtype=float)

# Fit
logreg_multi.fit(X_train_mc, y_multi_train)
rf_multi.fit(X_train_mc, y_multi_train)
ada_multi.fit(X_train_mc, y_multi_train, clf__sample_weight=sw_multi)

# Predict / evaluate
mc_results = []
mc_results.append(eval_multiclass("LogReg", y_multi_test, logreg_multi.predict(X_test_mc)))
mc_results.append(eval_multiclass("RandomForest", y_multi_test, rf_multi.predict(X_test_mc)))
mc_results.append(eval_multiclass("AdaBoost", y_multi_test, ada_multi.predict(X_test_mc)))

pd.DataFrame(mc_results)


## Results / Outputs
The cells above print per-model metric summaries and render confusion matrices. This section optionally adds DNN baselines when TensorFlow is available; otherwise it is skipped.


In [ ]:
# =====================
# Simple DNN (optional)
# =====================

def run_dnn_binary() -> dict | None:
    """Train/evaluate a lightweight DNN for the binary task (if TensorFlow is available)."""
    try:
        from tensorflow import keras
        from tensorflow.keras import layers
    except Exception:
        print("[INFO] TensorFlow/Keras not available, skipping DNN binary.")
        return None

    X_tr = lin_num_processor.fit_transform(X_train_bin)
    X_te = lin_num_processor.transform(X_test_bin)
    y_tr = y_bin_train.values.astype(np.int32)
    y_te = y_bin_test.values.astype(np.int32)

    model = keras.Sequential(
        [
            layers.Input(shape=(X_tr.shape[1],)),
            layers.Dense(128, activation="relu"),
            layers.Dropout(0.2),
            layers.Dense(64, activation="relu"),
            layers.Dense(1, activation="sigmoid"),
        ]
    )

    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(
        X_tr,
        y_tr,
        epochs=8,
        batch_size=1024,
        verbose=0,
        class_weight=class_weight_dict_bin,
    )

    proba = model.predict(X_te, verbose=0).ravel()
    pred = (proba >= 0.5).astype(int)
    return eval_binary("DNN", y_te, pred, proba)


def run_dnn_multiclass() -> dict | None:
    """Train/evaluate a lightweight DNN for the multiclass task (if TensorFlow is available)."""
    try:
        from tensorflow import keras
        from tensorflow.keras import layers
    except Exception:
        print("[INFO] TensorFlow/Keras not available, skipping DNN multiclass.")
        return None

    X_tr = lin_num_processor.fit_transform(X_train_mc)
    X_te = lin_num_processor.transform(X_test_mc)

    classes = np.unique(y_multi_train)
    class_to_idx = {c: i for i, c in enumerate(classes)}

    y_tr = np.array([class_to_idx[c] for c in y_multi_train.values], dtype=np.int32)
    y_te = np.array([class_to_idx[c] for c in y_multi_test.values], dtype=np.int32)
    k = len(classes)

    model = keras.Sequential(
        [
            layers.Input(shape=(X_tr.shape[1],)),
            layers.Dense(256, activation="relu"),
            layers.Dropout(0.3),
            layers.Dense(128, activation="relu"),
            layers.Dense(k, activation="softmax"),
        ]
    )

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    cw = {class_to_idx[c]: float(class_weight_dict_multi[c]) for c in classes}
    model.fit(X_tr, y_tr, epochs=12, batch_size=1024, verbose=0, class_weight=cw)

    pred_idx = model.predict(X_te, verbose=0).argmax(axis=1)
    pred = np.array([classes[i] for i in pred_idx])
    return eval_multiclass("DNN", y_multi_test.values, pred)


dnn_bin_res = run_dnn_binary()
dnn_mc_res = run_dnn_multiclass()

dnn_bin_res, dnn_mc_res


## Conclusions
- The notebook implements two supervised classification tasks over the same feature space, with a conservative leakage guard and numeric-only features.
- Linear and tree-based model families are evaluated under class imbalance via balanced weighting; AdaBoost receives explicit sample weights for consistency.
- Confusion matrices and macro-averaged reports support error analysis beyond overall accuracy.

**Next steps (typical extensions)**: feature engineering, hyperparameter tuning (e.g., nested CV), probability calibration for tree models, and per-class thresholding for operational deployment.
